# Open-World Re-ID Evaluation (Video-to-Video & Image-to-Video)

This notebook evaluates distance matrices generated by models in the **Open-World** setting.
It computes and prints:
- **Rank-1 DIR @ FAR = 1%, 5%, 10%** with 95% Bootstrap Confidence Intervals
- **Rank-5 DIR @ FAR = 1%, 5%, 10%** with 95% Bootstrap Confidence Intervals


In [5]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add root directory to sys.path
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR if (CURRENT_DIR / "evaluation").exists() else CURRENT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
if str(ROOT_DIR / "evaluation") not in sys.path:
    sys.path.append(str(ROOT_DIR / "evaluation"))

from evaluation.evaluation_utils import bootstrap_from_csv
print("Loaded evaluation_utils successfully!")
print(f"Project root: {ROOT_DIR}")


Loaded evaluation_utils successfully!
Project root: /d/hpc/projects/FRI/mm12755/DogReID-1553/DogReID-1553


In [6]:
# =====================================================================
# Configuration: Models and Candidate CSV Paths
# =====================================================================
MODELS = ["bot", "agw", "arbase", "sbs", "mgn", "tfclip", "clip", "baseline"]
MODALITIES = ["vid2vid", "img2vid"]

SEARCH_DIRS = [
    ROOT_DIR / "evaluation" / "csvs_open_results"
]

def find_open_csv(model_name: str, modality: str):
    """Find open-world distance matrix CSV for a given model and modality."""
    names = [model_name]
    if model_name == "baseline":
        names.extend(["dinov2", "dinov2_baseline"])
    elif model_name == "clip":
        names.extend(["clip_baseline", "clip_backbone"])

    for name in names:
        patterns = [
            f"{name}_{modality}/open_dist_matrix.csv",
            f"{name}_{modality}/vid2vid_open_dist_matrix.csv",
            f"{name}_{modality}/img2vid_open_dist_matrix.csv",
            f"{name}_open/open_dist_matrix.csv",
            f"{name}_open_{modality}/open_dist_matrix.csv",
        ]
        for search_dir in SEARCH_DIRS:
            if not search_dir.exists():
                continue
            for pat in patterns:
                candidate = search_dir / pat
                if candidate.exists() and candidate.stat().st_size > 1000:
                    return candidate
            # Direct folder check
            folder = search_dir / f"{name}_{modality}"
            if folder.exists():
                for f in folder.glob("*open*.csv"):
                    if f.stat().st_size > 1000:
                        return f
    return None

models_to_test = {}
for m in MODELS:
    for mod in MODALITIES:
        key = f"{m.upper()} ({mod})"
        csv_path = find_open_csv(m, mod)
        if csv_path:
            models_to_test[key] = str(csv_path)
            print(f"[FOUND] {key:<20} -> {csv_path.relative_to(ROOT_DIR)}")
        else:
            print(f"[PENDING] {key:<18} -> Not yet found in search directories")


[FOUND] BOT (vid2vid)        -> evaluation/csvs_open_results/bot_vid2vid/open_dist_matrix.csv
[FOUND] BOT (img2vid)        -> evaluation/csvs_open_results/bot_img2vid/open_dist_matrix.csv
[FOUND] AGW (vid2vid)        -> evaluation/csvs_open_results/agw_vid2vid/open_dist_matrix.csv
[FOUND] AGW (img2vid)        -> evaluation/csvs_open_results/agw_img2vid/open_dist_matrix.csv
[FOUND] ARBASE (vid2vid)     -> evaluation/csvs_open_results/arbase_vid2vid/open_dist_matrix.csv
[FOUND] ARBASE (img2vid)     -> evaluation/csvs_open_results/arbase_img2vid/open_dist_matrix.csv
[FOUND] SBS (vid2vid)        -> evaluation/csvs_open_results/sbs_vid2vid/open_dist_matrix.csv
[FOUND] SBS (img2vid)        -> evaluation/csvs_open_results/sbs_img2vid/open_dist_matrix.csv
[FOUND] MGN (vid2vid)        -> evaluation/csvs_open_results/mgn_vid2vid/open_dist_matrix.csv
[FOUND] MGN (img2vid)        -> evaluation/csvs_open_results/mgn_img2vid/open_dist_matrix.csv
[FOUND] TFCLIP (vid2vid)     -> evaluation/csvs_open_r

In [7]:
# =====================================================================
# Run Bootstrap Evaluation
# =====================================================================
M_ITERATIONS = 100
all_open_results = {}

for name, path in models_to_test.items():
    print("=" * 60)
    print(f"Evaluating {name} on {Path(path).name} ...")
    print("=" * 60)
    try:
        res = bootstrap_from_csv(path, m=M_ITERATIONS, mode="open")
        all_open_results[name] = res
    except Exception as e:
        print(f"[ERROR] Failed to evaluate {name}: {e}")


Evaluating BOT (vid2vid) on open_dist_matrix.csv ...
-> [CACHE HIT] Loaded precomputed open metrics for open_dist_matrix.csv (from bootstrap_cache_open_m100_seed42.pkl)
Evaluating BOT (img2vid) on open_dist_matrix.csv ...
-> [CACHE HIT] Loaded precomputed open metrics for open_dist_matrix.csv (from bootstrap_cache_open_m100_seed42.pkl)
Evaluating AGW (vid2vid) on open_dist_matrix.csv ...
-> [CACHE HIT] Loaded precomputed open metrics for open_dist_matrix.csv (from bootstrap_cache_open_m100_seed42.pkl)
Evaluating AGW (img2vid) on open_dist_matrix.csv ...
-> [CACHE HIT] Loaded precomputed open metrics for open_dist_matrix.csv (from bootstrap_cache_open_m100_seed42.pkl)
Evaluating ARBASE (vid2vid) on open_dist_matrix.csv ...
-> [CACHE HIT] Loaded precomputed open metrics for open_dist_matrix.csv (from bootstrap_cache_open_m100_seed42.pkl)
Evaluating ARBASE (img2vid) on open_dist_matrix.csv ...
-> [CACHE HIT] Loaded precomputed open metrics for open_dist_matrix.csv (from bootstrap_cache_op

In [8]:
# =====================================================================
# Display Open-World Benchmark Table (Rank-1 & Rank-5 DIR @ FAR = 1%, 5%, 10%)
# =====================================================================
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

rows = []
for name, res in all_open_results.items():
    full_fars = res["full_set"]["fars"]
    
    col_dict = {"Model (Modality)": name}
    for target, label in [(0.01, "FAR=1%"), (0.05, "FAR=5%"), (0.10, "FAR=10%")]:
        idx = np.argmin(np.abs(full_fars - target))
        
        # Rank-1 DIR (%)
        r1_val = res["dirs_r1_boot_mean"][idx] * 100
        r1_std = res.get("dirs_r1_std", np.zeros(len(full_fars)))[idx] * 100
        r1_l = res["dirs_r1_ci"][0][idx] * 100
        r1_u = res["dirs_r1_ci"][1][idx] * 100
        col_dict[f"R1 @ {label}"] = f"{r1_val:.1f} ± {r1_std:.1f} [{r1_l:.1f}-{r1_u:.1f}]%"
        
        # Rank-5 DIR (%)
        r5_val = res["dirs_r5_boot_mean"][idx] * 100
        r5_std = res.get("dirs_r5_std", np.zeros(len(full_fars)))[idx] * 100
        r5_l = res.get("dirs_r5_ci", res["dirs_r1_ci"])[0][idx] * 100
        r5_u = res.get("dirs_r5_ci", res["dirs_r1_ci"])[1][idx] * 100
        col_dict[f"R5 @ {label}"] = f"{r5_val:.1f} ± {r5_std:.1f} [{r5_l:.1f}-{r5_u:.1f}]%"
        
    rows.append(col_dict)

df_open = pd.DataFrame(rows)

print("=" * 185)
print(f"{'Model (Modality)':<20} | {'R1 @ FAR=1%':<25} | {'R5 @ FAR=1%':<25} | {'R1 @ FAR=5%':<25} | {'R5 @ FAR=5%':<25} | {'R1 @ FAR=10%':<25} | {'R5 @ FAR=10%':<25}")
print("=" * 185)
for r in rows:
    print(
        f"{r['Model (Modality)']:<20} | {r['R1 @ FAR=1%']:<25} | {r['R5 @ FAR=1%']:<25} | "
        f"{r['R1 @ FAR=5%']:<25} | {r['R5 @ FAR=5%']:<25} | {r['R1 @ FAR=10%']:<25} | {r['R5 @ FAR=10%']:<25}"
    )
print("-" * 185)

display(df_open)


Model (Modality)     | R1 @ FAR=1%               | R5 @ FAR=1%               | R1 @ FAR=5%               | R5 @ FAR=5%               | R1 @ FAR=10%              | R5 @ FAR=10%             
BOT (vid2vid)        | 3.2 ± 0.7 [2.0-4.6]%      | 3.6 ± 0.7 [2.4-5.1]%      | 6.9 ± 1.0 [5.2-8.4]%      | 9.0 ± 1.0 [7.2-11.1]%     | 10.8 ± 1.1 [9.0-13.0]%    | 14.3 ± 1.3 [12.3-16.9]%  
BOT (img2vid)        | 1.4 ± 0.3 [0.7-2.0]%      | 1.4 ± 0.4 [0.8-2.0]%      | 5.7 ± 0.8 [4.3-7.1]%      | 6.9 ± 0.8 [5.3-8.4]%      | 8.5 ± 1.0 [6.8-10.5]%     | 12.1 ± 1.1 [10.0-14.0]%  
AGW (vid2vid)        | 4.9 ± 0.7 [3.5-6.3]%      | 5.7 ± 0.8 [4.4-7.3]%      | 8.8 ± 1.0 [7.0-10.7]%     | 11.3 ± 1.0 [9.5-13.5]%    | 11.0 ± 1.0 [8.8-13.1]%    | 15.1 ± 1.0 [12.9-17.2]%  
AGW (img2vid)        | 2.6 ± 0.5 [1.7-3.6]%      | 3.0 ± 0.5 [2.1-4.0]%      | 5.4 ± 0.7 [4.0-6.8]%      | 7.0 ± 0.8 [5.6-8.4]%      | 8.9 ± 0.9 [7.3-10.8]%     | 12.6 ± 1.0 [10.4-14.6]%  
ARBASE (vid2vid)     | 4.4 ± 0.8 [3.1-6.2]%      | 5.3 

,Model (Modality),R1 @ FAR=1%,R5 @ FAR=1%,R1 @ FAR=5%,R5 @ FAR=5%,R1 @ FAR=10%,R5 @ FAR=10%
0,BOT (vid2vid),3.2 ± 0.7 [2.0-4.6]%,3.6 ± 0.7 [2.4-5.1]%,6.9 ± 1.0 [5.2-8.4]%,9.0 ± 1.0 [7.2-11.1]%,10.8 ± 1.1 [9.0-13.0]%,14.3 ± 1.3 [12.3-16.9]%
1,BOT (img2vid),1.4 ± 0.3 [0.7-2.0]%,1.4 ± 0.4 [0.8-2.0]%,5.7 ± 0.8 [4.3-7.1]%,6.9 ± 0.8 [5.3-8.4]%,8.5 ± 1.0 [6.8-10.5]%,12.1 ± 1.1 [10.0-14.0]%
2,AGW (vid2vid),4.9 ± 0.7 [3.5-6.3]%,5.7 ± 0.8 [4.4-7.3]%,8.8 ± 1.0 [7.0-10.7]%,11.3 ± 1.0 [9.5-13.5]%,11.0 ± 1.0 [8.8-13.1]%,15.1 ± 1.0 [12.9-17.2]%
3,AGW (img2vid),2.6 ± 0.5 [1.7-3.6]%,3.0 ± 0.5 [2.1-4.0]%,5.4 ± 0.7 [4.0-6.8]%,7.0 ± 0.8 [5.6-8.4]%,8.9 ± 0.9 [7.3-10.8]%,12.6 ± 1.0 [10.4-14.6]%
4,ARBASE (vid2vid),4.4 ± 0.8 [3.1-6.2]%,5.3 ± 0.8 [3.9-7.1]%,8.9 ± 1.1 [6.9-11.0]%,11.3 ± 1.2 [9.4-13.4]%,12.7 ± 1.3 [10.2-15.5]%,16.9 ± 1.4 [14.2-19.7]%
5,ARBASE (img2vid),2.8 ± 0.5 [1.8-3.8]%,3.4 ± 0.6 [2.4-4.6]%,7.1 ± 0.9 [5.4-9.1]%,9.6 ± 0.9 [8.2-11.6]%,10.8 ± 1.0 [9.2-13.0]%,14.6 ± 1.1 [12.7-17.1]%
6,SBS (vid2vid),4.7 ± 0.8 [3.3-6.4]%,5.5 ± 0.9 [3.9-7.3]%,10.1 ± 1.1 [7.8-12.4]%,12.2 ± 1.2 [9.9-14.5]%,14.9 ± 1.3 [12.2-17.6]%,19.7 ± 1.4 [16.8-22.3]%
7,SBS (img2vid),2.2 ± 0.5 [1.3-3.1]%,2.3 ± 0.5 [1.4-3.2]%,7.2 ± 0.8 [5.7-9.0]%,9.6 ± 1.0 [7.8-11.4]%,11.8 ± 1.1 [9.9-13.9]%,16.0 ± 1.2 [13.7-18.5]%
8,MGN (vid2vid),5.3 ± 0.9 [3.9-7.0]%,6.1 ± 0.9 [4.5-7.9]%,10.3 ± 1.2 [8.0-12.7]%,12.6 ± 1.1 [10.4-14.9]%,13.1 ± 1.3 [10.5-15.9]%,16.5 ± 1.2 [14.2-19.0]%
9,MGN (img2vid),2.5 ± 0.6 [1.6-3.6]%,2.9 ± 0.6 [2.0-3.9]%,7.5 ± 0.8 [6.0-9.3]%,9.4 ± 0.9 [7.9-11.2]%,9.3 ± 1.0 [7.3-11.2]%,13.0 ± 1.1 [10.9-15.2]%
